In [1]:
DB_PATH="db/universe.db"

In [2]:
# ── Cella 8: Funzioni RAG ──────────────────────────────────────
STOP_ITA = {"a","e","è","il","la","lo","le","i","gli","un","una",
             "di","da","in","su","per","con","che","si","mi","ci",
             "al","del","della","come","dove","cosa","non","ma","ho",
             "ha","sono","trova","sua","suo","questo","questa","quanto"}

def tokenize(q):
    toks = re.findall(r"[A-Za-z0-9àèéìòùÀÈÌÒÙ]+", q)
    f = [t for t in toks if len(t)>=3 and t.lower() not in STOP_ITA]
    return f if f else [t for t in toks if len(t)>=2]

def retrieve(query, db_path=DB_PATH, k=5):
    conn = sqlite3.connect(db_path); conn.row_factory=sqlite3.Row
    results=[]; seen=set()
    for tok in tokenize(query):
        rows = conn.execute(
            "SELECT * FROM galaxies WHERE name LIKE ? LIMIT 5",
            (f"%{tok}%",)).fetchall()
        for r in rows:
            d=dict(r)
            if d["name"] not in seen:
                seen.add(d["name"]); results.append(d)

    if not results:
        ql=query.lower()
        if any(x in ql for x in ["spiral","spirale"]):
            rows=conn.execute(
                "SELECT * FROM galaxies WHERE morpho_t BETWEEN 0 AND 7 ORDER BY mag_b ASC LIMIT ?",
                (k,)).fetchall()
        elif any(x in ql for x in ["ellittic","elliptical"]):
            rows=conn.execute(
                "SELECT * FROM galaxies WHERE morpho_t<=-4 ORDER BY mag_b ASC LIMIT ?",(k,)).fetchall()
        elif any(x in ql for x in ["vicin","near","prossim"]):
            rows=conn.execute(
                "SELECT * FROM galaxies WHERE redshift IS NOT NULL ORDER BY ABS(redshift) ASC LIMIT ?",(k,)).fetchall()
        elif any(x in ql for x in ["occhio","brillant","bright"]):
            rows=conn.execute(
                "SELECT * FROM galaxies WHERE mag_b IS NOT NULL ORDER BY mag_b ASC LIMIT ?",(k,)).fetchall()
        else:
            rows=conn.execute(
                "SELECT * FROM galaxies ORDER BY ABS(COALESCE(redshift,99)) ASC LIMIT ?",(k,)).fetchall()
        results=[dict(r) for r in rows]
    conn.close()
    print(results[:k])
    return results[:k]

def build_context(galaxies):
    if not galaxies: return "Nessuna galassia trovata nel database."
    lines=[]
    for g in galaxies:
        d=dm(g.get("redshift")); vel=g.get("velocity_km_s")
        line=f"• {g['name']} ({mstr(g.get('morpho_t'))})"
        if d: line+=f" — {d} Mpc"
        if vel:
            try: line+=f", v={float(vel):.0f} km/s"
            except: pass
        if g.get("mag_b"):
            try: line+=f", mag_B={float(g['mag_b']):.1f}"
            except: pass
        if g.get("distance_mpc"):
            try: line+=f", distance_mpc={float(g['distance_mpc']):.1f}"
            except: pass
        if g.get("note"): line+=f" | {g['note']}"
        line+=f" [fonte: {g.get('source','?')}]"
        lines.append(line)
    return " ".join(lines)

SYSTEM_PROMPT = (
    "Sei un astrofisico. Rispondi in italiano in modo chiaro e rigoroso. "
    "NON devi calcolare la distanza delle galassie: usa esclusivamente i valori forniti nei dati. "
    "Se local_group = 1, la distanza in distance_mpc in questo caso NON menzionare la legge di Hubble e NON usare H0. "
    "Se la velocità è negativa, significa che la galassia si sta avvicinando."
    "Non inventare formule o metodi non presenti nei dati. Rispetta il calcolo fatto è rispondi. "
    "Fornisci risposte che spiegano il perchè della stessa e  scientificamente corrette."
)

def build_prompt(query, galaxies):
    ctx = build_context(galaxies)
    return f"""{SYSTEM_PROMPT}

Dati dal database astronomico:
{ctx}

Domanda: {query}
Risposta:""".strip()

print("✓ Funzioni RAG definite")


✓ Funzioni RAG definite


In [3]:
# ── Cella C: caricamento modello ─────────────────────────────────
import IPython
#IPython.Application.instance().kernel.do_shutdown(True)
import gc, torch, kagglehub
from IPython.display import Markdown, display
from transformers import AutoProcessor, AutoModelForImageTextToText

gc.collect()
torch.cuda.empty_cache()

MODEL_PATH = kagglehub.model_download("google/gemma-4/transformers/gemma-4-e2b-it")

processor = AutoProcessor.from_pretrained(MODEL_PATH)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.bfloat16,
    device_map="auto",
).eval()
print(f"✓ Modello caricato su: {model.device}")

# ── Funzione RAG ─────────────────────────────────────────────────
def ask_gemma(query, db_path=DB_PATH, max_new_tokens=400):
    galaxies = retrieve(query, db_path)
    prompt   = build_prompt(query, galaxies)

    messages = [{
        "role": "user",
        "content": [{"type": "text", "text": prompt}]
    }]

    print('Dati: ',prompt)
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        add_generation_prompt=True,
        enable_thinking=False,
    ).to(model.device)

    input_len = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=1.0,
            top_p=0.95,
            top_k=64,
            do_sample=True,
        )

    return processor.decode(outputs[0][input_len:], skip_special_tokens=True)

# ── Test ─────────────────────────────────────────────────────────
#for q in test_queries[:3]:
#    print(f"Q: {q}")
#    display(Markdown(f"**A:**\n\n{ask_gemma(q)}\n"))

ModuleNotFoundError: No module named 'torch'

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

# --- Elementi dell'Interfaccia ---
title = widgets.HTML("<h2>AstroEdge Explorer</h2><p>Chiedi a Gemma 4 informazioni sulle galassie nel database locale.</p>")
query_input = widgets.Text(
    placeholder='Es: "Quali galassie ellittiche ci sono?" o "Parlami di Andromeda"',
    description='Domanda:',
    layout=widgets.Layout(width='70%')
)
run_button = widgets.Button(
    description='Interroga Gemma 4',
    button_style='primary', # 'success', 'info', 'warning', 'danger' or ''
    tooltip='Clicca per avviare il ragionamento del modello',
    icon='rocket'
)
out = widgets.Output()

# --- Logica di Esecuzione ---
def on_click_callback(b):
    with out:
        clear_output()
        user_query = query_input.value
        if not user_query.strip():
            print("Per favore, inserisci una domanda.")
            return
        
        print(f"🔍 Analisi in corso per: '{user_query}'...")
        print("Wait... Gemma 4 sta interrogando il database locale...")
        
        try:
            # Chiama la tua funzione RAG esistente
            response = ask_gemma(user_query)
            
            # Mostra il risultato in Markdown
            display(Markdown(f"### Risposta di Gemma 4:\n---\n{response}"))
            
        except Exception as e:
            print(f"Errore durante l'elaborazione: {e}")

# Collegamento del pulsante alla funzione
run_button.on_click(on_click_callback)

# Visualizzazione dell'interfaccia
display(title)
display(widgets.HBox([query_input, run_button]))
display(out)